# 📥 Bulk PPTX Downloader for Google Colab

Downloads PPT/PPTX files from HuggingFace datasets, filters out:
- Chinese content (filename + slide content scan)
- USA-specific content (filename + slide content scan)
- Files under 2MB
- Empty/corrupted files

Clean files are saved directly to your Google Drive.

**Datasets used:**
1. `noxneural/pptx_collection_templates` — ~2,000 PPTX templates (20 GB)
2. `F171636/pptx_info` — Additional PPTX files

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Create output folders on Google Drive
DRIVE_BASE = '/content/drive/MyDrive/PPTX_Dataset'
CLEAN_DIR = os.path.join(DRIVE_BASE, 'clean_pptx')
REJECTED_DIR = os.path.join(DRIVE_BASE, 'rejected')
STATS_DIR = os.path.join(DRIVE_BASE, 'stats')

os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(REJECTED_DIR, exist_ok=True)
os.makedirs(STATS_DIR, exist_ok=True)

print(f'✅ Google Drive mounted')
print(f'📁 Clean files:    {CLEAN_DIR}')
print(f'📁 Rejected files: {REJECTED_DIR}')

## Step 2: Install Dependencies

In [ ]:
!pip install -q huggingface_hub

## Step 3: Download from HuggingFace

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

# Download Dataset 1: noxneural/pptx_collection_templates (~20 GB, ~2000 files)
print('📥 Downloading noxneural/pptx_collection_templates...')
print('   This is ~20 GB — may take 15-30 minutes on Colab...')

local_path_1 = snapshot_download(
    repo_id='noxneural/pptx_collection_templates',
    repo_type='dataset',
    allow_patterns=['*.pptx', '*.ppt'],
    local_dir='/content/hf_download_1',
)

files_1 = list(Path(local_path_1).rglob('*.pptx')) + list(Path(local_path_1).rglob('*.ppt'))
print(f'✅ Dataset 1: {len(files_1)} files downloaded')

# Download Dataset 2: F171636/pptx_info
print('\n📥 Downloading F171636/pptx_info...')
try:
    local_path_2 = snapshot_download(
        repo_id='F171636/pptx_info',
        repo_type='dataset',
        allow_patterns=['*.pptx', '*.ppt'],
        local_dir='/content/hf_download_2',
    )
    files_2 = list(Path(local_path_2).rglob('*.pptx')) + list(Path(local_path_2).rglob('*.ppt'))
    print(f'✅ Dataset 2: {len(files_2)} files downloaded')
except Exception as e:
    print(f'⚠️ Dataset 2 failed: {e}')
    files_2 = []

all_files = files_1 + files_2
print(f'\n📂 Total files to process: {len(all_files)}')

## Step 4: Define Filters

In [ ]:
import re
import zipfile
import hashlib
import shutil
import json
from datetime import datetime

MIN_SIZE = 2 * 1024 * 1024  # 2 MB

# --- China/HK/Taiwan keywords ---
CHINA_KEYWORDS = [
    'chinese new year', 'china ', 'hong kong', 'taiwan',
    'beijing', 'shanghai', 'mandarin', 'cantonese',
    'spring festival', 'mid-autumn', 'dragon boat',
]

# --- USA keywords (only strong indicators) ---
USA_KEYWORDS = [
    'american government', 'american history', 'american revolution',
    'us constitution', 'bill of rights', 'us election',
    'us president', 'pledge of allegiance', 'state of the union',
    'fourth of july', '4th of july', 'veterans day',
    'memorial day', 'presidents day', 'martin luther king',
]

def is_china_filename(name):
    lower = name.lower()
    return any(kw in lower for kw in CHINA_KEYWORDS)

def is_usa_filename(name):
    lower = name.lower()
    return any(kw in lower for kw in USA_KEYWORDS)

def has_chinese_content(filepath):
    """Check if PPTX slide content has significant Chinese characters."""
    try:
        with zipfile.ZipFile(filepath, 'r') as z:
            for name in z.namelist():
                if 'slide1.xml' in name:
                    data = z.read(name).decode('utf-8', errors='ignore')
                    cn_chars = len(re.findall(r'[\u4e00-\u9fff]', data))
                    if cn_chars > 50:
                        return True
                    break
    except:
        pass
    return False

def is_valid_pptx(filepath):
    """Check magic bytes: PK (ZIP/PPTX) or D0CF (old PPT)."""
    try:
        with open(filepath, 'rb') as f:
            header = f.read(8)
        return header[:2] == b'PK' or header[:8] == b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1'
    except:
        return False

def file_hash(filepath):
    """SHA1 hash for dedup."""
    h = hashlib.sha1()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()[:12]

print('✅ Filters defined')

## Step 5: Filter & Copy Clean Files to Google Drive

In [ ]:
from IPython.display import clear_output

stats = {
    'total': len(all_files),
    'too_small': 0,
    'invalid': 0,
    'china_filename': 0,
    'china_content': 0,
    'usa_filename': 0,
    'duplicate': 0,
    'kept': 0,
    'errors': 0,
}

seen_hashes = set()

# Also track existing files in Drive to avoid re-copying
existing_files = set(os.listdir(CLEAN_DIR)) if os.path.exists(CLEAN_DIR) else set()
print(f'📂 Already in Drive: {len(existing_files)} files\n')

for i, f in enumerate(all_files, 1):
    if i % 50 == 0 or i == len(all_files):
        clear_output(wait=True)
        pct = i / len(all_files) * 100
        print(f'🔍 Processing: {i}/{len(all_files)} ({pct:.1f}%)')
        print(f'   ✅ Kept: {stats["kept"]}  |  ❌ Rejected: {i - stats["kept"] - stats["errors"]}')
        print(f'   📊 Small: {stats["too_small"]} | Invalid: {stats["invalid"]} | China: {stats["china_filename"]+stats["china_content"]} | USA: {stats["usa_filename"]} | Dup: {stats["duplicate"]}')

    reason = None
    fp = Path(f)

    try:
        # Size check
        sz = fp.stat().st_size
        if sz < MIN_SIZE:
            reason = 'too_small'
            stats['too_small'] += 1

        # Format check
        if not reason and not is_valid_pptx(fp):
            reason = 'invalid'
            stats['invalid'] += 1

        # China filename check
        if not reason and is_china_filename(fp.name):
            reason = 'china_filename'
            stats['china_filename'] += 1

        # USA filename check
        if not reason and is_usa_filename(fp.name):
            reason = 'usa_filename'
            stats['usa_filename'] += 1

        # Dedup via content hash
        if not reason:
            fh = file_hash(fp)
            if fh in seen_hashes:
                reason = 'duplicate'
                stats['duplicate'] += 1
            else:
                seen_hashes.add(fh)

        # Deep Chinese content check
        if not reason and has_chinese_content(fp):
            reason = 'china_content'
            stats['china_content'] += 1

        # Copy to appropriate destination
        if reason:
            # Move to rejected subfolder
            rej_dir = os.path.join(REJECTED_DIR, reason)
            os.makedirs(rej_dir, exist_ok=True)
            dest = os.path.join(rej_dir, fp.name)
            if not os.path.exists(dest):
                shutil.copy2(str(fp), dest)
        else:
            # Clean file → save to Drive
            dest = os.path.join(CLEAN_DIR, fp.name)
            if not os.path.exists(dest):
                shutil.copy2(str(fp), dest)
            stats['kept'] += 1

    except Exception as e:
        stats['errors'] += 1

print('\n' + '=' * 60)
print('📊 FINAL FILTERING RESULTS')
print('=' * 60)
print(f'   Total files scanned:    {stats["total"]}')
print(f'   ❌ Under 2MB:           {stats["too_small"]}')
print(f'   ❌ Invalid format:       {stats["invalid"]}')
print(f'   ❌ China (filename):     {stats["china_filename"]}')
print(f'   ❌ China (content):      {stats["china_content"]}')
print(f'   ❌ USA (filename):       {stats["usa_filename"]}')
print(f'   ❌ Duplicate:            {stats["duplicate"]}')
print(f'   ⚠️  Errors:              {stats["errors"]}')
print(f'   ✅ KEPT (clean):         {stats["kept"]}')
print('=' * 60)
print(f'\n✅ Clean files saved to: {CLEAN_DIR}')

## Step 6: Save Stats Report

In [ ]:
# Save stats to a JSON file on Drive
stats_file = os.path.join(STATS_DIR, f'filter_stats_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json')
with open(stats_file, 'w') as f:
    json.dump(stats, f, indent=2)

# Count final files in clean folder
final_count = len([f for f in os.listdir(CLEAN_DIR) if f.endswith(('.ppt', '.pptx'))])
total_size_gb = sum(
    os.path.getsize(os.path.join(CLEAN_DIR, f))
    for f in os.listdir(CLEAN_DIR)
    if f.endswith(('.ppt', '.pptx'))
) / (1024**3)

print(f'📊 Stats saved to: {stats_file}')
print(f'\n🎉 FINAL SUMMARY:')
print(f'   📁 Clean files in Drive: {final_count}')
print(f'   💾 Total size: {total_size_gb:.2f} GB')
print(f'   📍 Location: Google Drive > PPTX_Dataset > clean_pptx')

## Step 7: Cleanup Colab Storage (Optional)
Run this to free up Colab disk space after files are saved to Drive.

In [ ]:
# Uncomment to delete local downloads (saves Colab space)
# !rm -rf /content/hf_download_1 /content/hf_download_2
# print('🗑️ Local downloads cleaned up. Files are safe on Google Drive.')